# Distributed Training Profiling and Debugging

## Overview

Tools and techniques for profiling and debugging distributed training.

### Topics Covered
- PyTorch Profiler
- NCCL debugging
- Memory profiling
- Common issues

## 1. PyTorch Profiler

In [ ]:
import torch
from torch.profiler import profile, ProfilerActivity, tensorboard_trace_handler

def profile_training_step(model, data, target, optimizer, criterion):
    """Profile a training step with PyTorch Profiler."""
    
    with profile(
        activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
        record_shapes=True,
        profile_memory=True,
        with_stack=True,
    ) as prof:
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
    
    # Print summary
    print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))
    return prof

## 2. Memory Profiling

In [ ]:
def memory_stats():
    """Print GPU memory statistics."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        max_allocated = torch.cuda.max_memory_allocated() / 1e9
        
        print(f"Allocated: {allocated:.2f} GB")
        print(f"Reserved:  {reserved:.2f} GB")
        print(f"Peak:      {max_allocated:.2f} GB")
    else:
        print("CUDA not available")

memory_stats()

## 3. NCCL Debugging

```bash
# Enable NCCL debug logging
export NCCL_DEBUG=INFO
export NCCL_DEBUG_SUBSYS=ALL

# Common issues:
# - NCCL timeout: Check network connectivity
# - Hang at init: Firewall blocking ports
# - Slow AllReduce: Check InfiniBand config
```

## 4. Common Issues

| Issue | Symptom | Solution |
|-------|---------|----------|
| OOM | CUDA out of memory | Reduce batch size, use gradient checkpointing |
| Deadlock | Training hangs | Check all ranks execute same ops |
| Slow | Low GPU utilization | Profile communication overhead |
| NaN loss | Training diverges | Check learning rate, gradient clipping |